In [1]:
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor




In [3]:
# Molecular Data
mol_df = pl.read_csv("../data/raw/X_train/molecular_train.csv")
mol_eval = pl.read_csv("../data/raw/X_test/molecular_test.csv" , ignore_errors=True)

# TRAITEMENT DE MOLECULAR DATA SET


In [4]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0
…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null


In [5]:
mol_df.describe()

statistic,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,str,f64,f64,str,str,str,str,str,f64,f64
"""count""","""10935""","""10821""",10821.0,10821.0,"""10821""","""10821""","""10935""","""10923""","""10935""",10846.0,10821.0
"""null_count""","""0""","""114""",114.0,114.0,"""114""","""114""","""0""","""12""","""0""",89.0,114.0
"""mean""",null,null,8.0783e7,8.0783e7,null,null,null,null,null,0.305087,1051.229554
"""std""",null,null,5.6427e7,5.6427e7,null,null,null,null,null,0.211524,552.861902
"""min""","""P100000""","""1""",394899.0,394899.0,"""A""","""A""","""ABL1""","""FLT3_ITD""","""2KB_upstream_variant""",0.02,16.0
"""25%""",null,null,3.1022441e7,3.1022442e7,null,null,null,null,null,0.1025,660.0
"""50%""",null,null,7.4732959e7,7.4732959e7,null,null,null,null,null,0.3213,975.0
"""75%""",null,null,1.15258747e8,1.15258747e8,null,null,null,null,null,0.442,1353.0
"""max""","""P132729""","""X""",2.26252135e8,2.26252135e8,"""TTTTCA""","""TTTTC""","""ZRSR2""","""p.Y974fs*8""","""synonymous_codon""",0.999,7156.0


In [6]:
mol_df.null_count()

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,114,114,114,114,114,0,12,0,89,114


# VAF , DEPTH TREATMENT

In [7]:
quant_vars = ["VAF" , "DEPTH"]
sub_df = mol_df.select(quant_vars)

# Conver to pandas
sub_pd = sub_df.to_pandas()

# Apply Model-Based Imputation (Random Forest)
imputer = IterativeImputer(estimator=RandomForestRegressor(), random_state=42)
imputed_data = imputer.fit_transform(sub_pd)

# 4. To Polars
imputed_pl = pl.DataFrame(imputed_data, schema=sub_df.columns)

# 5. Replace nulls
mol_df = mol_df.with_columns([imputed_pl[col].alias(col) for col in quant_vars])

c:\Users\zakar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [9]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0
…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583


In [8]:
mol_df["CHR"].value_counts().to_numpy()


array([['5', 369],
       ['7', 537],
       ['10', 18],
       ['3', 115],
       ['8', 41],
       ['14', 4],
       ['4', 1699],
       ['12', 596],
       ['11', 326],
       ['9', 116],
       ['15', 192],
       ['21', 866],
       ['22', 55],
       ['20', 999],
       [None, 114],
       ['1', 365],
       ['X', 1204],
       ['13', 42],
       ['19', 155],
       ['18', 128],
       ['16', 91],
       ['17', 1387],
       ['6', 3],
       ['2', 1513]], dtype=object)

In [10]:
def to_int(str):
    if(str == 'X'):
        return 23 # A justifier ?
    else:
        return int(str)

mol_df = mol_df.with_columns(
    pl.col("CHR").map_elements(to_int , return_dtype=pl.Int64)
    .alias("CHR")
)



In [12]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,i64,f64,f64,str,str,str,str,str,f64,f64
"""P100000""",11,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0
"""P100000""",5,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0
"""P100000""",3,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0
"""P100000""",4,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0
"""P100000""",2,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0
…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.2014,811.120583


# ORDINAL ENCODING

In [13]:
def dna_to_array(dna):
    lst = []
    
    nitrogen_bases = ['a','g','t','c']
    dna = dna.lower()
    
    
    for ch in dna:
        if(ch in nitrogen_bases):
            lst.append(ch)
        else:
            lst.append('n')
        
        
    return lst

def ordinal_encoder_dna(dna):
    mapping = {'a' : 0.25 , 'c' : 0.5 ,'g':0.75 , 't' : 1.00}
    
    nitrogen_bases = ['a','g','t','c']
    
    return [mapping[x] if x in nitrogen_bases else 0.00 for x in dna ]

    


seq_test = 'TTCAGCCAGTG'
ordinal_encoder_dna(dna_to_array(seq_test))





[1.0, 1.0, 0.5, 0.25, 0.75, 0.5, 0.5, 0.25, 0.75, 1.0, 0.75]

# K-MER ENCODING

In [14]:
def extract_kmers(sequence, k):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]


def join_str(str):
    return ' '.join(str)



k = 7 # K-mer
mySeq= 'TCTCACACATGTGCCAATCACTGTCACCC' # DNA Sequence


print(f"Example of transformation of DNA sequence : {mySeq} with {k}-mer")
new_str = join_str(extract_kmers(mySeq, k=7))

print(f"Transformation : {new_str} ")

Example of transformation of DNA sequence : TCTCACACATGTGCCAATCACTGTCACCC with 7-mer
Transformation : TCTCACA CTCACAC TCACACA CACACAT ACACATG CACATGT ACATGTG CATGTGC ATGTGCC TGTGCCA GTGCCAA TGCCAAT GCCAATC CCAATCA CAATCAC AATCACT ATCACTG TCACTGT CACTGTC ACTGTCA CTGTCAC TGTCACC GTCACCC 


In [15]:
from sklearn.feature_extraction.text import CountVectorizer


mySeq= 'TCTCACACATGTGCCAATCACTGTCACCC'
mySeq2 = 'GTGCCCAGGTTCAGTGAGTGACACAGGCAG'


DNA_list = [join_str(extract_kmers(mySeq,k=6)) , join_str(extract_kmers(mySeq2,k=6))]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(DNA_list).toarray()

print("Vocabulary: ", vectorizer.vocabulary_)


print(f"Encoded DNA : {X}")




Vocabulary:  {'tctcac': 41, 'ctcaca': 24, 'tcacac': 37, 'cacaca': 13, 'acacat': 2, 'cacatg': 15, 'acatgt': 4, 'catgtg': 20, 'atgtgc': 11, 'tgtgcc': 47, 'gtgcca': 34, 'tgccaa': 44, 'gccaat': 28, 'ccaatc': 21, 'caatca': 12, 'aatcac': 0, 'atcact': 10, 'tcactg': 39, 'cactgt': 16, 'actgtc': 5, 'ctgtca': 25, 'tgtcac': 46, 'gtcacc': 31, 'tcaccc': 38, 'gtgccc': 35, 'tgccca': 45, 'gcccag': 29, 'cccagg': 23, 'ccaggt': 22, 'caggtt': 18, 'aggttc': 7, 'ggttca': 30, 'gttcag': 36, 'ttcagt': 48, 'tcagtg': 40, 'cagtga': 19, 'agtgag': 9, 'gtgagt': 33, 'tgagtg': 43, 'gagtga': 27, 'agtgac': 8, 'gtgaca': 32, 'tgacac': 42, 'gacaca': 26, 'acacag': 1, 'cacagg': 14, 'acaggc': 3, 'caggca': 17, 'aggcag': 6}
Encoded DNA : [[1 0 1 0 1 1 0 0 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 1 1 0 0 1 0 0 1 0 0 1 0
  0 1 1 1 0 1 0 0 1 0 1 1 0]
 [0 1 0 1 0 0 1 1 1 1 0 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 0 1 1 0 1
  1 0 0 0 1 0 1 1 0 1 0 0 1]]


In [ ]:
# TREAT THE REF with length <= 4 (arbitrary ?) and >4 differently ! 


k_mer_coef  = 4

mol_df = mol_df.with_columns(
    pl.when(pl.col("REF").str.len_chars() <= k_mer_coef)
    .then(pl.col("REF"))
    .otherwise(None) 
    .alias("REF_not_mer")
)


